# Parsimony — flagship training run

Trains a ~50M-parameter language model **from scratch** for GIBC V2 Track 01.

**Run this on Kaggle** (Settings → Accelerator → **GPU T4 x2**). Kaggle gives 30 GPU-hours/week
and 12-hour sessions, which beats Colab's free tier for this. Every stage checkpoints and
resumes, so a disconnect costs you nothing — just re-run the same cell.

**Run as two separate "Save Version → Save & Run All (Commit)" runs, not one long interactive session.**
Each stage is independent by design (see the module docstring in `scripts/run_gpu.py`), so a
12-hour session limit — or your computer going to sleep — never costs you the whole pipeline:

1. **Commit run 1** — set `MODE = "sweep"` below. Runs data + sweep only, ~1.5–2.5 hours at
   the default `--sweep-flops`. Produces `parsimony_sweep_output.zip` in the output panel.
2. Add that zip as a new Kaggle **Dataset**, then attach it as input data to a new notebook version.
3. **Commit run 2** — set `MODE = "flagship"` below and `SWEEP_OUTPUT_DIR` to that dataset's
   `/kaggle/input/...` path. Skips data + sweep, copies their output back into place, and trains
   only the flagship model.


## 0 · Setup

Replace `REPO_URL` with your GitHub repo once you've pushed the code.

In [ ]:
REPO_URL = "https://github.com/1234620/parsimony.git"   # <-- edit this

MODE = "sweep"   # "sweep" for commit run 1 (data + sweep). Switch to "flagship" for commit run 2.
SWEEP_OUTPUT_DIR = "/kaggle/input/parsimony-sweep-output"   # only read when MODE == "flagship" —
                                                             # point this at the dataset holding run 1's zip, unzipped

import os, subprocess, sys
if not os.path.exists("parsimony"):
    subprocess.run(["git", "clone", REPO_URL, "parsimony"], check=True)
os.chdir("parsimony")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "datasets"], check=True)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("MODE =", MODE)

if MODE == "flagship":
    import shutil
    assert os.path.exists(SWEEP_OUTPUT_DIR), (
        f"SWEEP_OUTPUT_DIR not found: {SWEEP_OUTPUT_DIR}. "
        "Add run 1's output zip as a Kaggle Dataset, attach it to this notebook as input data, "
        "and point SWEEP_OUTPUT_DIR at its path (check the Input panel on the right for the exact path)."
    )
    shutil.copytree(os.path.join(SWEEP_OUTPUT_DIR, "data"), "data", dirs_exist_ok=True)
    shutil.copytree(os.path.join(SWEEP_OUTPUT_DIR, "results"), "results", dirs_exist_ok=True)
    print("Copied sweep output from", SWEEP_OUTPUT_DIR)


## 1 · Verify the parameter budget

Track 01 requires a printed parameter count and model config. This is that artifact — and it
proves every configuration sits under the 50,000,000 cap *including* embeddings.

In [ ]:
!python scripts/verify_params.py

## 2 · Stage `data` — build the corpus and tokenizers

Mixture: TinyStories (fluency at small scale), FineWeb-Edu (clean world knowledge), and
procedurally generated worked-reasoning traces (step-by-step arithmetic and transitive logic).
If a HuggingFace source is unavailable the stage warns and continues rather than dying.

**~20–35 minutes.** Tokenizer training on several vocabulary sizes is the slow part.

In [ ]:
if MODE == "sweep":
    !python scripts/run_gpu.py --stage data \
        --docs 400000 --synth-docs 120000 --tok-docs 120000 \
        --vocabs 2048 8192 16384 32768 50257
else:
    print("Skipping data stage — carried over from the sweep run's output.")


## 3 · Stage `sweep` — compute-matched vocabulary study

The core experiment. Each configuration gets the **same parameter budget** and the **same FLOP
budget**; only the vocabulary (and therefore the depth it leaves room for) changes. Selects the
winning vocabulary by bits-per-byte and writes `results/selected_vocab.json`.

**~1.5–2.5 hours.** Reduce `--sweep-flops` to 1e16 if you're short on GPU quota.

In [ ]:
if MODE == "sweep":
    !python scripts/run_gpu.py --stage sweep \
        --vocabs 2048 8192 16384 32768 50257 \
        --sweep-budget 12000000 --sweep-dmodel 256 --sweep-flops 6e15 \
        --block 512 --batch 32
else:
    print("Skipping sweep stage — carried over from the sweep run's output.")


In [ ]:
if MODE == "sweep":
    import json
    rows = json.load(open("results/sweep_gpu/results.json"))
    print(f"{'vocab':>7} {'L':>3} {'emb%':>6} {'B/tok':>7} {'BPB':>8} {'tokPPL':>9} {'reason':>7}")
    for r in rows:
        print(f"{r['vocab_actual']:>7} {r['n_layers']:>3} {r['embed_fraction']:>5.1%} "
              f"{r['bytes_per_token']:>7.2f} {r['bits_per_byte']:>8.4f} "
              f"{r['token_ppl']:>9.2f} {r['reasoning_acc']:>7.1%}")


## 4 · Stage `flagship` — train the full ~50M model

Trains at the vocabulary the sweep selected. Checkpoints every 500 steps to `runs/flagship/` —
if the session dies, re-run this exact cell and it resumes from the last checkpoint.

**~3–4 hours** at the settings below. Watch the first few hundred steps: loss should fall from
about `ln(vocab)` steadily. If it plateaus above 6.0 or produces NaN, halve `--lr` and re-run.

In [ ]:
if MODE == "flagship":
    !python scripts/run_gpu.py --stage flagship \
        --dmodel 512 --block 512 --batch 24 --grad-accum 2 \
        --steps 12000 --lr 1.5e-3
else:
    print('Skipping flagship — this is the sweep run. Re-run this notebook with MODE = "flagship" as commit run 2.')


## 5 · Results and sample generations

`results/flagship.json` records hardware, wall-clock and total FLOPs — all three are required
in the README and *training efficiency is a scored criterion*, so don't skip this cell.

In [ ]:
if MODE == "flagship":
    import json
    print(json.dumps(json.load(open("results/flagship.json")), indent=2))


In [ ]:
if MODE == "flagship":
    import sys, torch, json
    sys.path.insert(0, "src")
    from model import Parsimony, ParsimonyConfig
    from tokenizer_train import load_tokenizer, EOS_ID
    from pathlib import Path

    ck = torch.load("runs/flagship/ckpt.pt", map_location="cuda", weights_only=False)
    cfg = ParsimonyConfig(**ck["model_cfg"])
    model = Parsimony(cfg).cuda(); model.load_state_dict(ck["model"]); model.eval()
    tok = load_tokenizer(sorted(Path("data").glob("tok_*.json"))[0])

    for prompt in ["Question: There are 6 boxes with 7 apples in each. How many apples in total?\nStep 1:",
                   "Once upon a time there was a small",
                   "Question: Ana has 24 coins. Ben has 11 coins. Who has more?\nStep 1:"]:
        ids = torch.tensor([tok.encode(prompt).ids]).cuda()
        out = model.generate(ids, max_new_tokens=48, temperature=0.7, top_k=40, eos_id=EOS_ID)
        print(repr(prompt))
        print("  ->", tok.decode(out[0, ids.shape[1]:].tolist()).strip()[:200], "\n")


## 6 · Save artifacts

Download `parsimony_artifacts.zip` from the Kaggle output panel, then commit `results/` to the
repo and attach the checkpoint to a GitHub Release (it's too large for git itself).

In [ ]:
if MODE == "sweep":
    !zip -qr /kaggle/working/parsimony_sweep_output.zip data results
    !ls -lh /kaggle/working/parsimony_sweep_output.zip
    print("Add this zip as a new Kaggle Dataset, then use it as input data for the MODE='flagship' commit run.")
else:
    !zip -qr /kaggle/working/parsimony_artifacts.zip results runs/flagship/ckpt.pt data/tok_*.json
    !ls -lh /kaggle/working/parsimony_artifacts.zip
